# Sierra Support — a guardrailed customer-service agent (Sierra-style)

[Sierra](https://sierra.ai) builds production customer-service agents that
actually *resolve* issues — looking up orders, issuing refunds, updating
records — against real business systems, under strict policy, with clean
escalation to humans when they reach their limits.

What makes Sierra-style agents trustworthy is **architecture**, not just
prompting. Three load-bearing ideas:

```
customer ──▶ support agent ──▶ business-system tools ──▶ action (refund / update / ...)
                │               └── policy in the CODE       │
                │                   (hard limits that          └── audit log (every action,
                │                    can't be argued around)        recorded in Python, always)
                └──▶ escalate_to_human
                     (structured ticket — never improvise)
```

**SDK primitives this notebook showcases:**
- `@tool` for grounded business-system actions with code-level policy enforcement
- `AuditLogMiddleware` — built-in structured logging of every agent lifecycle event
- `TokenBudgetMiddleware`, `CostBudgetMiddleware` — cost control per support session
- `ToolPolicyMiddleware` — whitelist exactly which tools this agent may call
- Custom `AgentMiddleware` — a lightweight session-cost tracker showing how to write your own

> **Before running:** copy `.env.example` to `.env` at the repo root and add your provider key.

## 1. Environment & constants

| Constant | What it controls |
|---|---|
| `PROVIDER` / `MODEL` | The LLM that powers the support agent. |
| `REFUND_LIMIT_USD` | Hard ceiling on automatic refunds. Anything above this is rejected in Python code and escalated — the model cannot argue around it. |
| `SESSION_TOKEN_BUDGET` | Max tokens per support session. `TokenBudgetMiddleware` aborts if exceeded — prevents runaway conversations. |
| `SESSION_COST_BUDGET_USD` | Dollar cap per session. A second financial guard on top of the token cap. |

In [ ]:
%pip install -q vidbyte-sdk python-dotenv

import json
import logging
import os
from datetime import datetime, timezone

from dotenv import load_dotenv

load_dotenv()
load_dotenv("../../.env")

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL    = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")
REFUND_LIMIT_USD        = 100.00   # hard ceiling — enforced in Python, not the prompt
SESSION_TOKEN_BUDGET    = 15_000
SESSION_COST_BUDGET_USD = 0.25

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + overrides) in .env first."

## 2. System prompt

The system prompt is the **first policy layer** — it encodes tone, scope
limits, identity-verification protocol, and when to escalate. The second
policy layer lives in the tool code (next section) and holds even if the
model is social-engineered past the prompt.

Key things to adapt for your own deployment:
- **Identity verification** — the `find_orders(email=...)` call is the minimal
  form. Replace with session auth, account ID lookup, etc.
- **Scope** — add domain restrictions ("only handle orders placed in the last
  90 days") to match your policy.
- **Escalation SLA** — change "one business day" to your actual SLA.

In [ ]:
SYSTEM_PROMPT = """You are a customer support agent for Summit Outfitters.

Policy:
- Identity first: ask for the customer's email, call find_orders, and confirm
  the order before discussing it. Never reveal another customer's data.
- Grounded actions only: never state outcomes you have not confirmed with a tool.
  If a tool returns an error or a refund is refused, tell the customer exactly why
  and call escalate_to_human — do not retry or promise alternative outcomes.
- Scope: order status, refunds, and shipping questions only. Anything outside
  this scope gets escalated immediately.
- Tone: warm, direct, no corporate filler. One question at a time."""

## 3. Tools

The tools are the agent's **only window into reality** — it cannot know
anything about an order that `find_orders` or `get_order` didn't return, and
it cannot change anything that `process_refund` didn't confirm.

Notice `process_refund`: the `REFUND_LIMIT_USD` check is a Python `if` — not
a prompt instruction. The model can be persuaded; the `if` cannot. This is
the second policy layer, and it is the more reliable one.

Every `process_refund` call — approved or rejected — is appended to
`AUDIT_LOG`. This happens in Python, unconditionally, before the function
returns. The agent cannot suppress it.

In [ ]:
from vidbyte import tool

# ── In-memory business systems (replace with real DB calls in production) ──
ORDERS = {
    "A-1001": {"email": "dana@example.com", "item": "Trail Running Shoes",   "amount": 89.00,  "status": "delivered", "refunded": False},
    "A-1002": {"email": "dana@example.com", "item": "Carbon Trekking Poles", "amount": 240.00, "status": "delivered", "refunded": False},
    "A-1003": {"email": "sam@example.com",  "item": "Headlamp",              "amount": 35.00,  "status": "shipped",   "refunded": False},
}
AUDIT_LOG: list[dict]   = []
ESCALATIONS: list[dict] = []


def _audit(action: str, **kw):
    AUDIT_LOG.append({"ts": datetime.now(timezone.utc).isoformat(), "action": action, **kw})


@tool
def find_orders(email: str) -> list[dict]:
    """Find all orders for a customer email address. Returns order_id, item,
    amount, status, and refund state for each order found."""
    _audit("find_orders", email=email)
    return [
        {"order_id": oid, **{k: v for k, v in o.items() if k != "email"}}
        for oid, o in ORDERS.items() if o["email"].lower() == email.lower()
    ]


@tool
def get_order(order_id: str) -> dict:
    """Fetch the details of one order by its order_id."""
    _audit("get_order", order_id=order_id)
    order = ORDERS.get(order_id)
    return {"order_id": order_id, **order} if order else {"error": f"Order {order_id!r} not found."}


@tool
def process_refund(order_id: str, reason: str) -> dict:
    """Issue a full refund for an order. Succeeds only when: the order exists,
    has not already been refunded, and the amount is at or below the
    auto-refund limit. Returns ok=True on success, ok=False with a reason
    on failure — the caller must escalate on failure."""
    order = ORDERS.get(order_id)
    if order is None:
        _audit("refund_rejected", order_id=order_id, why="unknown order")
        return {"ok": False, "why": f"Order {order_id!r} not found."}
    if order["refunded"]:
        _audit("refund_rejected", order_id=order_id, why="already refunded")
        return {"ok": False, "why": "This order has already been refunded."}
    if order["amount"] > REFUND_LIMIT_USD:
        _audit("refund_rejected", order_id=order_id, why="over auto-refund limit",
               amount=order["amount"], limit=REFUND_LIMIT_USD)
        return {"ok": False,
                "why": f"Amount ${order['amount']:.2f} exceeds the ${REFUND_LIMIT_USD:.2f} "
                       "auto-refund limit. Please escalate to a human agent."}
    order["refunded"] = True
    _audit("refund_processed", order_id=order_id, amount=order["amount"], reason=reason)
    return {"ok": True, "refunded_amount": order["amount"]}


@tool
def escalate_to_human(customer_email: str, summary: str, severity: str) -> dict:
    """Open a support ticket for a human agent. severity must be 'low', 'normal',
    or 'high'. Call whenever a request is outside policy or your tools cannot
    resolve it — never improvise when escalation is the right answer."""
    ticket = {
        "ticket_id": f"T-{len(ESCALATIONS)+1:04d}",
        "customer_email": customer_email,
        "summary": summary,
        "severity": severity if severity in ("low", "normal", "high") else "normal",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    ESCALATIONS.append(ticket)
    _audit("escalated", **ticket)
    return ticket

## 4. Middleware

Four middleware layers, each guarding a different risk:

| Middleware | What it guards |
|---|---|
| `AuditLogMiddleware` | **Observability** — emits a structured log event at every lifecycle hook (before/after model call, before/after tool call). Essential for compliance review and debugging in production. |
| `TokenBudgetMiddleware` | **Cost** — aborts if the session exceeds `SESSION_TOKEN_BUDGET`. Prevents a determined customer from running up a large bill by keeping the agent in an endless loop. |
| `CostBudgetMiddleware` | **Cost** — dollar-level cap on top of the token cap. |
| `ToolPolicyMiddleware` | **Scope** — whitelists the exact tool names this agent may call. Even if a future code change adds new tools to the function list, `ToolPolicyMiddleware` won't allow them unless explicitly added here. |

We also define a **custom `SessionCostTrackerMiddleware`** to show how easy
it is to extend the middleware system — this one logs cumulative token usage
after every model call.

In [ ]:
import logging

from vidbyte.lib.dataclasses.middleware import MiddlewareContext, MiddlewareDecision
from vidbyte.middleware import (
    AgentMiddleware,
    AuditLogMiddleware,
    CostBudgetMiddleware,
    TokenBudgetMiddleware,
    ToolPolicyMiddleware,
)


class SessionCostTrackerMiddleware(AgentMiddleware):
    """Custom middleware: prints a one-line token-usage summary after each model call."""
    name = "SessionCostTracker"

    async def after_model_response(self, ctx: MiddlewareContext) -> MiddlewareDecision:
        tokens = ctx.tokens_used or 0
        print(f"  [tokens used so far: {tokens:,}]")
        return MiddlewareDecision.continue_()


middleware = [
    AuditLogMiddleware(logger=logging.getLogger("support"), log_level=logging.INFO),
    TokenBudgetMiddleware(max_tokens=SESSION_TOKEN_BUDGET),
    CostBudgetMiddleware(max_cost_usd=SESSION_COST_BUDGET_USD),
    ToolPolicyMiddleware(allowed_tools=["find_orders", "get_order", "process_refund", "escalate_to_human"]),
    SessionCostTrackerMiddleware(),
]

## 5. Constructing the agent

The support agent uses the default **linear runtime** — one model call per
iteration, tools called in sequence. That's correct here: customer service
is a synchronous back-and-forth, not a parallel research task.

`PermissionPolicy` defaults to `SAFE + READ`, which would block our `@tool`
functions (they're `SAFE` by default from the decorator, so actually fine
here — but we use `allow_all()` explicitly to be clear and future-proof).

Agent history persists across `run()` calls, so each cell below is one
conversational turn in the same session.

In [ ]:
from vidbyte import BaseAgent
from vidbyte.tools.security import PermissionPolicy

support = BaseAgent(
    name="support-agent",
    system_prompt=SYSTEM_PROMPT,
    provider=PROVIDER,
    model_name=MODEL,
    tools=[find_orders, get_order, process_refund, escalate_to_human],
    middleware=middleware,
    permission_policy=PermissionPolicy.allow_all(),
    max_tool_rounds=6,
)

## 6. Running the agent — three example scenarios

Each cell below is one turn. The agent remembers the full conversation, so
turns build on each other within the same session.

**Scenario A** — happy path: an in-policy refund. Dana's $89 shoes are under
the auto-refund limit, so the agent should verify identity, find the order,
and resolve it without involving a human.

**Scenario B** — guardrail path: over-limit refund. The $240 trekking poles
exceed `REFUND_LIMIT_USD`. Watch the layers work: the agent *tries* the
refund, the Python code refuses it, and the prompt policy routes the result
into a structured escalation ticket.

**Other scenarios to try:**
```
"I never received my order, it says delivered but I don't have it."
"Can you update my shipping address? My order hasn't shipped yet."   # out of scope → escalate
"I want to return both items I ordered."                             # multi-item session
```

In [ ]:
# Scenario A: happy path — in-policy refund
reply = support.run(
    "Hi, I'd like a refund on my running shoes. My email is dana@example.com."
)
print(reply.content)

In [ ]:
# Scenario B: guardrail path — over-limit refund (triggers code-level policy + escalation)
reply = support.run("Can you also refund my trekking poles?")
print(reply.content)

## 7. Example output

### Scenario A — happy path
```
To confirm your identity, I've looked up your orders for dana@example.com
and found order A-1001: Trail Running Shoes, $89.00, delivered.

I've processed a full refund of $89.00 for order A-1001. You should see
the credit in 3-5 business days. Is there anything else I can help you with?
```

### Scenario B — guardrail path
```
I attempted the refund for order A-1002 (Carbon Trekking Poles, $240.00),
but it exceeds our $100.00 automatic refund limit.

I've opened ticket T-0001 for a human agent who will follow up within
one business day to complete this for you. Your case reference is T-0001.
```

### Audit log (inspect after both scenarios)

In [ ]:
# Inspect the paper trail — every action recorded in Python, not by the model
print("AUDIT LOG")
for entry in AUDIT_LOG:
    print(" ", json.dumps(entry))

print("\nESCALATION QUEUE")
for ticket in ESCALATIONS:
    print(" ", json.dumps(ticket))

print("\nORDER STATE")
for oid, order in ORDERS.items():
    print(f"  {oid}: refunded={order['refunded']}")